# 📘 Colab: Auto-Flatten Nested JSON (Mixed-Type Arrays)

In [ ]:

# ✅ Colab Notebook: Auto-Flatten Nested JSON with Mixed-Type Arrays

!pip install glom

import json
import pandas as pd
import numpy as np
from glom import glom, PathAccessError
from pprint import pprint

# Load JSON
with open("/content/deep_nested_users.json") as f:
    data = json.load(f)

# Automatically collect all field paths (recursively), handling mixed arrays
def collect_paths(obj, current_path=None):
    if current_path is None:
        current_path = []
    paths = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            paths += collect_paths(v, current_path + [k])
    elif isinstance(obj, list):
        if len(obj) > 0:
            # Check if elements are dicts
            if all(isinstance(item, dict) for item in obj):
                # Use the first dict to get keys
                for k in obj[0].keys():
                    paths.append(current_path + ["[]", k])
            else:
                # Mixed type (e.g., list of strings, numbers)
                paths.append(current_path + ["[]"])
        else:
            # Empty list
            paths.append(current_path)
    else:
        paths.append(current_path)
    return paths

# Convert path list to glom-style path string
def to_glom_path(path_list):
    result = []
    for item in path_list:
        if item == "[]":
            result.append("[]")
        else:
            result.append(str(item))
    return ".".join(result)

# Build schema from first record
schema_paths = collect_paths(data[0])
schema = {}
for path_list in schema_paths:
    key_name = "_".join([p if p != "[]" else "list" for p in path_list])
    schema[key_name] = to_glom_path(path_list)

# Safe extract using glom
def safe_glom(record, schema):
    result = {}
    for key, path in schema.items():
        try:
            result[key] = glom(record, path)
        except PathAccessError:
            result[key] = np.nan
    return result

# Flatten all records
flattened = [safe_glom(record, schema) for record in data]
df = pd.DataFrame(flattened)

# Automatically expand all list columns, even if mixed types
def auto_expand_lists(df):
    list_cols = df.columns[df.applymap(type).eq(list).any()]
    for col in list_cols:
        max_len = df[col].map(lambda x: len(x) if isinstance(x, list) else 0).max()
        for i in range(max_len):
            df[f"{col}_{i+1}"] = df[col].apply(
                lambda x: x[i] if isinstance(x, list) and len(x) > i else np.nan
            )
        df.drop(columns=[col], inplace=True)
    return df

df = auto_expand_lists(df)
df
